# Depth Anything V2 (Low Compute) on Kaggle

This notebook runs your `baseline/depth_anything_v2` CLI with a low-compute setup:

- Model: `vits` (small)
- Input size: `384`
- Optional image cap (default `32`)
- CPU-safe by default (`--device auto`)

## Kaggle Datasets to attach
1. **Code dataset (required)**: upload this repo as a Kaggle Dataset, then attach it.
2. **Image dataset (required)**: recommended small option for smooth runs:
   - https://www.kaggle.com/datasets/ifigotin/imagenetmini-1000


## 1) Verify runtime dependencies
Kaggle images usually include these already. If any import fails, enable Internet and install manually.


In [ ]:
import importlib

required = ['numpy', 'cv2', 'torch', 'torchvision', 'matplotlib']
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    raise RuntimeError(
        'Missing packages: ' + ', '.join(missing) + '. ' \
        'Enable Kaggle Internet and install them, e.g. !pip install ' + ' '.join(missing)
    )

import numpy as np, cv2, torch, torchvision, matplotlib
print('numpy', np.__version__)
print('opencv', cv2.__version__)
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('matplotlib', matplotlib.__version__)


## 2) Configure dataset paths
Set `CODE_DATASET_DIRNAME` to your attached code dataset folder name in `/kaggle/input`.


In [ ]:
from pathlib import Path
import os

KAGGLE_INPUT = Path('/kaggle/input')
assert KAGGLE_INPUT.exists(), 'This notebook expects Kaggle runtime (/kaggle/input).'

# Change this to your code dataset folder name if needed.
CODE_DATASET_DIRNAME = 'eaglevision-code'

# Optional hint for your image dataset folder name (can be None).
IMAGE_DATASET_HINT = 'imagenetmini-1000'

print('Attached datasets:')
for p in sorted(KAGGLE_INPUT.iterdir()):
    if p.is_dir():
        print('-', p.name)


## 3) Copy code to writable workspace and import
Kaggle input datasets are read-only, so we copy code to `/kaggle/working`.


In [ ]:
import shutil
import sys

source_repo = KAGGLE_INPUT / CODE_DATASET_DIRNAME
if not source_repo.exists():
    raise FileNotFoundError(
        f'Code dataset not found: {source_repo}. Update CODE_DATASET_DIRNAME.'
    )

work_repo = Path('/kaggle/working/eaglevision')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(source_repo, work_repo)

sys.path.insert(0, str(work_repo))
os.chdir(work_repo)

print('Repo ready at:', work_repo)


## 4) Discover image dataset and create a small subset
To keep compute low, we infer on up to `MAX_IMAGES` files.


In [ ]:
import itertools

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
MAX_IMAGES = 32  # lower for cheaper runs

def score_dataset_dir(path: Path) -> tuple[int, int]:
    score = 0
    if IMAGE_DATASET_HINT and IMAGE_DATASET_HINT.lower() in path.name.lower():
        score += 10
    image_count = 0
    for child in path.rglob('*'):
        if child.is_file() and child.suffix.lower() in IMAGE_SUFFIXES:
            image_count += 1
            if image_count >= 200:
                break
    if image_count > 0:
        score += 1
    return score, image_count

candidates = []
for ds in KAGGLE_INPUT.iterdir():
    if not ds.is_dir() or ds.name == CODE_DATASET_DIRNAME:
        continue
    score, image_count = score_dataset_dir(ds)
    if image_count > 0:
        candidates.append((score, image_count, ds))

if not candidates:
    raise RuntimeError('No image datasets found in /kaggle/input. Attach one image dataset.')

candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
image_dataset_root = candidates[0][2]

all_images = [
    p for p in image_dataset_root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
]
all_images = sorted(all_images)
if not all_images:
    raise RuntimeError(f'No image files found in {image_dataset_root}')

subset_dir = Path('/kaggle/working/low_compute_subset')
if subset_dir.exists():
    shutil.rmtree(subset_dir)
subset_dir.mkdir(parents=True, exist_ok=True)

for src in all_images[:MAX_IMAGES]:
    rel = src.relative_to(image_dataset_root)
    dst = subset_dir / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print('Selected image dataset:', image_dataset_root)
print('Total discovered images:', len(all_images))
print('Subset size:', len(list(subset_dir.rglob('*'))))
print('Subset root:', subset_dir)


## 5) Download only small (`vits`) relative checkpoint
Saved to writable location in `/kaggle/working/da_checkpoints`.


In [ ]:
CHECKPOINT_DIR = Path('/kaggle/working/da_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

!python -m baseline.depth_anything_v2 download \
    --mode relative \
    --encoder vits \
    --checkpoints-dir /kaggle/working/da_checkpoints


## 6) Run low-compute inference


In [ ]:
OUTPUT_DIR = Path('/kaggle/working/depth_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

!python -m baseline.depth_anything_v2 infer \
    --input /kaggle/working/low_compute_subset \
    --output-dir /kaggle/working/depth_outputs \
    --mode relative \
    --encoder vits \
    --input-size 384 \
    --device auto \
    --checkpoints-dir /kaggle/working/da_checkpoints


## 7) Preview output images


In [ ]:
import matplotlib.pyplot as plt

preview_paths = sorted(OUTPUT_DIR.rglob('*_depth.png'))[:12]
print('Depth PNG outputs:', len(preview_paths))

if preview_paths:
    cols = 3
    rows = (len(preview_paths) + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 3 * rows))
    for i, p in enumerate(preview_paths, start=1):
        img = plt.imread(p)
        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


## 8) Optional: zip outputs for download


In [ ]:
import shutil

zip_base = '/kaggle/working/depth_outputs'
archive_path = shutil.make_archive(zip_base, 'zip', str(OUTPUT_DIR))
print('Saved:', archive_path)
